# Local Campaign Build Test

Step-by-step smoke test for `LocalCampaign`.  
Each cell is independent — run until one throws, paste the traceback.

**Before running Cell 5 (live devices):** disconnect DDS and Ultra syringes from drive screws.

Required env vars:
- `XPD_DATA_SESSION` — active NSLS-II proposal, e.g. `pass-314159`
- `TILED_BLUESKY_WRITING_API_KEY_XPD` — Tiled write key (for Cells 4+)

## Cell 1 — load config, inspect connection block

In [ ]:
import json
import os
from pprint import pprint

CONFIG_PATH = "halide_local.json"

cfg = json.load(open(CONFIG_PATH))
pprint(cfg["connection"])
print()
print("tiled_profile    :", cfg["connection"]["tiled_profile"])
print("tiled_catalog_path:", cfg["connection"].get("tiled_catalog_path", "(absent — will default to ['xpd','raw'])"))
print("API key set      :", bool(os.environ.get("TILED_BLUESKY_WRITING_API_KEY_XPD")))
print("DATA_SESSION     :", os.environ.get("XPD_DATA_SESSION", "(not set)"))

## Cell 2 — build agent from config only (no Tiled, no hardware)

In [ ]:
from xpd_tools.optimization.agent import BuildAgent

agent = BuildAgent.from_config(cfg)

print("queue_server      :", agent.queue_server)
print("tiled_profile     :", agent.tiled_profile)
print("tiled_catalog_path:", agent.tiled_catalog_path)
print("evaluation_method :", agent.evaluation_method)
print("dofs              :", agent.dofs)
print("sources           :", [s.dof for s in agent.sources] if agent.sources else None)
print("run.iterations    :", agent.run_settings.iterations if agent.run_settings else None)

## Cell 3 — test Tiled client resolution alone

Exercises `_tiled_client()`: `from_profile(tiled_profile)["xpd"]["raw"]` + QoS header.  
`KeyError: 'xpd'` here = profile already resolves to the xpd node (double-index).  
Fix: set `tiled_profile: "nsls2"` and `tiled_catalog_path: ["xpd", "raw"]` in JSON (already done).

In [ ]:
client = agent._tiled_client()
print("client type:", type(client))
print("client     :", client)
print("qos header :", client.context.http_client.headers.get("tiled-qos"))

## Cell 4 — build full LocalCampaign with mock devices (no hardware, no motion)

Exercises the full build graph: Tiled write client, RunEngine, simulated devices, blop agent.  
Requires `XPD_DATA_SESSION` in env.

In [ ]:
from xpd_tools.optimization.local import LocalCampaign

DATA_SESSION = os.environ.get("XPD_DATA_SESSION", "")
assert DATA_SESSION, "Set XPD_DATA_SESSION env var first (e.g. export XPD_DATA_SESSION=pass-314159)"

campaign = LocalCampaign.from_config(
    cfg,
    data_session=DATA_SESSION,
    mock=True,   # simulated devices — no EPICS, no motion
)

print("RE              :", campaign.RE)
print("RE.md           :", dict(campaign.RE.md))
print("writing_client  :", campaign.writing_client)
print("reading_client  :", campaign.reading_client)
print("devices         :", list(campaign.devices))
print("run_agent       :", campaign.run_agent)

## Cell 5 — check device connections (live EPICS, no motion)

Builds LocalCampaign with real devices.  
No motion: `check_connections` only reads PV status, does not command anything.  
Skip if not on beamline network.

In [ ]:
live = LocalCampaign.from_config(
    cfg,
    data_session=DATA_SESSION,
    mock=False,
)

reports = live.check_connections(require_alive=False)  # False = warn, not raise
for r in reports:
    status = "OK" if r.connected else "FAIL"
    alive  = r.alive if r.alive is not None else "n/a"
    print(f"  [{status}] {r.name:20s}  connected={r.connected}  alive={alive}")
    if r.failed_signals:
        print(f"           failed signals: {r.failed_signals}")
    if r.liveness_detail:
        print(f"           liveness: {r.liveness_detail}")

## Cell 6 — single mock trial (simulated acquisition, no hardware)

Runs `sample_suggestions` through the local RE against mock devices.  
Verifies the full plan→evaluate→log round-trip without touching hardware.

In [ ]:
point = {
    "infusion_rate_CsPb": 20.0,
    "infusion_rate_Br":   20.0,
    "infusion_rate_Cl":   20.0,
    "infusion_rate_I":    20.0,
}

# campaign from Cell 4 (mock=True)
result = campaign.single_trial(point)
print("result:", result)

## Cell 7 — summarise (after at least one trial)

Returns a DataFrame of all trials so far.

In [ ]:
df = campaign.summarize()
display(df)